In [ ]:
"""
Imports,
Tokenizer,
Tokenization,
Dataset split,
collate_fn definition,
DataLoaders,
Model definition,
Device setup.
Optimizer,
Training loop,
Validation function,
Convergence detection,
Baseline snapshot,
Compression step,
Recovery training loop
"""

In [ ]:
# First, mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math
from transformers import AutoTokenizer
from datasets import Dataset
import pandas as pd
from copy import deepcopy

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/openhermes.csv')
display(df.head())

,instruction,input,output
0,Write a Perl script that processes a log file ...,NaN,```perl\n#!/usr/bin/perl\n\nuse strict;\nuse w...
1,"What can be seen once in a minute, twice in a ...",NaN,The letter 'M'.
2,Famous inventors and their inventions: Identif...,NaN,1. Thomas Edison: One of his most significant ...
3,Generate a list of 12 words that start with 'qu'.,NaN,1. Quail\n2. Quarry\n3. Quasar\n4. Quench\n5. ...
4,"Who was the first woman to win a Nobel Prize, ...",NaN,Marie Curie; Physics


In [ ]:
df['input'] = df['input'].fillna("")
df = df.dropna(subset=['output'])

def format_row(row):
    if row["input"].strip() == "":
        return f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['output']}"
    else:
        return f"### Instruction: /n{row['instruction']}\n\n### Input:\n{row['input']}\n\n### Response:\n {row['output']}"

df['text'] = df.apply(format_row, axis=1)
display(df['text'].head())

,text
0,### Instruction:\nWrite a Perl script that pro...
1,### Instruction:\nWhat can be seen once in a m...
2,### Instruction:\nFamous inventors and their i...
3,### Instruction:\nGenerate a list of 12 words ...
4,### Instruction:\nWho was the first woman to w...


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = Dataset.from_pandas(df[['text']])
dataset = dataset.map(tokenize, batched=True)
dataset

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/242818 [00:00<?, ? examples/s]

Dataset({
    features: ['text', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 242818
})

In [ ]:
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch])
    attention_mask = torch.tensor([item["attention_mask"] for item in batch])
    return input_ids, attention_mask

# loader = DataLoader(dataset, batch_size=32 ,shuffle=True, collate_fn=collate_fn)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=4,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        seq_len = x.size(1)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        ).bool()

        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

In [ ]:
class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, d_ff=1024, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = tokenizer.vocab_size
model = DecoderOnlyLM(vocab_size, d_model=256, d_ff=1024)
model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

def train_one_epoch(model, train_loader, optimizer, vocab_size, device):
    model.train()
    total_loss = 0

    for input_ids, attention_mask in train_loader:
        input_ids = input_ids.to(device)
        logits = model(input_ids)

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        loss = F.cross_entropy(
            shift_logits.view(-1, vocab_size),
            shift_labels.view(-1),
            ignore_index=tokenizer.pad_token_id
        )
        optimizer.zero_grad()
        loss.backward()
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2

        total_norm = total_norm ** 0.5
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
# for epoch in range(10):
#     for step, (input_ids, attention_mask) in enumerate(train_loader):
#         input_ids = input_ids.to(device)
#         attention_mask = attention_mask.to(device)
#         logits = model(input_ids)

#         shift_logits = logits[:, :-1, :].contiguous()
#         shift_labels = input_ids[:, 1:].contiguous()

#         loss = F.cross_entropy(
#             shift_logits.view(-1, vocab_size),
#             shift_labels.view(-1),
#             ignore_index=tokenizer.pad_token_id
#         )
#         optimizer.zero_grad()
#         loss.backward()
#         total_norm = 0
#         for p in model.parameters():
#             if p.grad is not None:
#                 param_norm = p.grad.data.norm(2)
#                 total_norm += param_norm.item() ** 2

#         total_norm = total_norm ** 0.5
#         optimizer.step()

#         if step % 100 == 0:
#             print(f"Epoch: {epoch}, Step: {step}, Loss: {loss.item():.4f}")

In [ ]:
# Validation function
def evaluate(model, val_loader, vocab_size):
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for input_ids, attention_mask in val_loader:
            input_ids = input_ids.to(device)
            logits = model(input_ids)

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, vocab_size),
                shift_labels.view(-1),
                ignore_index=tokenizer.pad_token_id,
                reduction='sum'
            )

            total_loss += loss.item()
            non_pad = (shift_labels != tokenizer.pad_token_id).sum().item()
            total_tokens += non_pad
    model.train()
    return total_loss / total_tokens

In [ ]:
def train_until_convergence(model, train_loader, val_loader, optimizer, vocab_size, device, max_epochs=50, patience=3):
    best_val_loss = float('inf')
    epoch_no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, vocab_size, device)
        val_loss = evaluate(model, val_loader, vocab_size)

        print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

        if val_loss < best_val_loss * 0.995:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epoch_no_improve = 0
        else:
            epoch_no_improve += 1

        if epoch_no_improve >= patience:
            print("Converged")
            break
    model.load_state_dict(best_state)
    return best_val_loss

In [ ]:
# best_val_loss = float('inf')
# epoch_no_improve = 0
# patience = 3

# for epoch in range(max_epochs):
#     train_one_epoch()
#     val_loss = evaluate(model, test_loader, vocab_size)

#     if val_loss < best_val_loss * 0.995:
#         best_val_loss = val_loss
#         epoch_no_improve = 0
#     else:
#         epoch_no_improve += 1

#     if epoch_no_improve >= patience:
#         print("Converged")
#         break

In [ ]:
# baseline_val_loss = None

In [ ]:
baseline_val_loss = train_until_convergence(model, train_loader, test_loader, optimizer, vocab_size, device)
print(f"Baseline Validation Loss: {baseline_val_loss}")

In [ ]:
def compress_model_ffn(model, compression_ratio=0.1):
    new_d_ff = int(model.blocks[0].ffn[0].out_features * (1 - compression_ratio))
    print(f"Compressing to {new_d_ff}")
    new_model = DecoderOnlyLM(
        vocab_size,
        d_model=256,
        d_ff=new_d_ff,
        n_layers=len(model.blocks)
    ).to(device)

    # copy embedding + lm_head
    new_model.embed.load_state_dict(model.embed.state_dict())
    new_model.lm_head.load_state_dict(model.lm_head.state_dict())
    new_model.norm.load_state_dict(model.norm.state_dict())

    for old_block, new_block in zip(model.blocks, new_model.blocks):
        # copy attention + norms
        new_block.attn.load_state_dict(old_block.attn.state_dict())
        new_block.norm1.load_state_dict(old_block.norm1.state_dict())
        new_block.norm2.load_state_dict(old_block.norm2.state_dict())

        # FFN pruning
        W1 = old_block.ffn[0].weight.data
        b1 = old_block.ffn[0].bias.data
        W2 = old_block.ffn[2].weight.data
        b2 = old_block.ffn[2].bias.data

        # compute L2 norm of each neuron
        neuron_importance = torch.norm(W2, dim=0)

        # get top-k neurons to keep
        keep_indices = torch.topk(neuron_importance, new_d_ff).indices
        keep_indices, _ = torch.sort(keep_indices)

        # prune W1 rows
        new_block.ffn[0].weight.data.copy_(W1[keep_indices, :])
        new_block.ffn[0].bias.data.copy_(b1[keep_indices])

        # prune W2 columns
        new_block.ffn[2].weight.data.copy_(W2[:, keep_indices])
        new_block.ffn[2].bias.data.copy_(b2)

    return new_model

In [ ]:
def recover_after_compression(model, train_loader, val_loader, optimizer, baseline_val_loss, vocab_size, device, max_steps=3000):
    train_iter = iter(train_loader)

    for step in range(max_steps):
        try:
            input_ids, _ = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            input_ids, _ = next(train_iter)

        input_ids = input_ids.to(device)
        logits = model(input_ids)
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        loss = F.cross_entropy(
            shift_logits.view(-1, vocab_size),
            shift_labels.view(-1),
            ignore_index=tokenizer.pad_token_id,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 200 == 0:
            val_loss = evaluate(model, test_loader, vocab_size)
            print(f"Recovery Val Loss: {val_loss}")

            if val_loss <= baseline_val_loss * 1.01:
                recovered = True
                print("Recovered")
                return True
    print("Compression Failed!")
    return False

In [ ]:
current_model = model
current_baseline = baseline_val_loss

while True:
    compressed_model = compress_model_ffn(current_model, 0.1)
    optimizer = torch.optim.AdamW(compressed_model.parameters(), lr=3e-4)
    success = recover_after_compression(compressed_model, train_loader, test_loader, optimizer, current_baseline, vocab_size, device)
    if not success:
        break

current_model = compressed_model
current_baseline = train_until_convergence(current_model, train_loader, test_loader, optimizer, vocab_size, device, max_epochs=5)

In [ ]:
# max_recovery_steps = 3000
# recovered = False

# for recovery_step in range(max_recovery_steps):
#     input_ids, attention_mask = next(iter(train_loader))
#     input_ids = input_ids.to(device)
#     attention_mask = attention_mask.to(device)

#     logits = model(input_ids)
#     shift_logits = logits[:, :-1, :].contiguous()
#     shift_labels = input_ids[:, 1:].contiguous()

#     loss = F.cross_entropy(
#         shift_logits.view(-1, vocab_size),
#         shift_labels.view(-1),
#         ignore_index=tokenizer.pad_token_id,
#     )

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if recovery_step % 200 == 0:
#         val_loss = evaluate(model, test_loader, vocab_size)
#         print(f"Recovery Val Loss: {val_loss}")

#         if val_loss <= baseline_val_loss * 1.01:
#             recovered = True
#             print("Recovered")
#             break

#     # input_ids, attention_mask = next(iter(train_loader))

# if not recovered:
#     print("Compression Failed!")

In [ ]:
# total_norm = 0
# for p in model.parameters():
#     if p.grad is not None:
#         param_norm = p.grad.data.norm(2)
#         total_norm += param_norm.item() ** 2

# total_norm = total_norm ** 0.5

In [ ]:
final_val_loss = evaluate(current_model, test_loader, vocab_size)
perplexity = math.exp(final_val_loss)

In [ ]:
# vocab_size = 1000
# seq_len = 16
# batch_size = 32

# model = TinyModel(vocab_size)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# for step in range(200):
#     x = torch.randint(0, vocab_size, (batch_size, seq_len))
#     y = torch.randint(0, vocab_size, (batch_size, seq_len))

#     logits = model(x)
#     loss = F.cross_entropy(
#         logits.view(-1, vocab_size),
#         y.view(-1)
#     )

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if step % 20 == 0:
#         print(f"Step: {step}, Loss: {loss.item():.4f}")
